In [13]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [14]:
ratings_path = "data/ratings.dat"

ratings = pd.read_csv(
    ratings_path,
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"],
    usecols=[0, 1, 2],
)

ratings.head()

,user_id,movie_id,rating
0,1,1193,5
1,1,661,3
2,1,914,3
3,1,3408,4
4,1,2355,5


In [15]:
print("ratings:", ratings.shape)
print(ratings["rating"].describe())
print("unique users:", ratings["user_id"].nunique())
print("unique movies:", ratings["movie_id"].nunique())

total_possible = ratings["user_id"].nunique() * ratings["movie_id"].nunique()
print("sparsity:", 1 - (len(ratings) / total_possible))

ratings: (1000209, 3)
count    1.000209e+06
mean     3.581564e+00
std      1.117102e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64
unique users: 6040
unique movies: 3706
sparsity: 0.9553163743776871


In [16]:
user_ids = ratings["user_id"].unique()
movie_ids = ratings["movie_id"].unique()

user2idx = {int(u): i for i, u in enumerate(user_ids)}
movie2idx = {int(m): i for i, m in enumerate(movie_ids)}

df = pd.DataFrame({
    "user_idx": ratings["user_id"].map(user2idx).astype("int64"),
    "movie_idx": ratings["movie_id"].map(movie2idx).astype("int64"),
    "rating": ratings["rating"].astype("float32"),
})

df.head()

,user_idx,movie_idx,rating
0,0,0,5.0
1,0,1,3.0
2,0,2,3.0
3,0,3,4.0
4,0,4,5.0


In [17]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42
)

train_df.head()

,user_idx,movie_idx,rating
416292,2506,76,2.0
683230,4086,897,4.0
2434,18,151,3.0
688533,4117,7,4.0
472584,2906,518,4.0


In [21]:
class RatingsDataset(Dataset):
    def __init__(self, df):
        self.u = torch.tensor(df["user_idx"].values, dtype=torch.long)
        self.m = torch.tensor(df["movie_idx"].values, dtype=torch.long)
        self.y = torch.tensor(df["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.u[i], self.m[i], self.y[i]

class MF(nn.Module):
    def __init__(self, n_users, n_movies, k=32):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, k)
        self.movie_emb = nn.Embedding(n_movies, k)
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_bias = nn.Embedding(n_movies, 1)

    def forward(self, user_idx, movie_idx):
        u = self.user_emb(user_idx)
        m = self.movie_emb(movie_idx)
        dot = (u * m).sum(dim=1)
        b = (self.user_bias(user_idx) + self.movie_bias(movie_idx)).squeeze(1)
        return dot + b


In [22]:
device = "cuda" if torch.cuda.is_available() else "cpu"

n_users = int(df["user_idx"].nunique())
n_movies = int(df["movie_idx"].nunique())

model = MF(n_users, n_movies, k=32).to(device)

train_loader = DataLoader(
    RatingsDataset(train_df), batch_size=4096, shuffle=True
)
test_loader = DataLoader(
    RatingsDataset(test_df), batch_size=4096, shuffle=False
)

optimizer = torch.optim.Adam(
    model.parameters(), lr=1e-2, weight_decay=1e-5
)
loss_fn = nn.MSELoss()


In [24]:
def rmse_on(loader):
    model.eval()
    se = 0.0
    n = 0
    with torch.no_grad():
        for u, m, y in loader:
            u, m, y = u.to(device), m.to(device), y.to(device)
            pred = model(u, m)
            se += ((pred - y) ** 2).sum().item()
            n += y.numel()
    return (se / n) ** 0.5

for epoch in range(1, 10):
    model.train()
    for u, m, y in train_loader:
        u, m, y = u.to(device), m.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(u, m)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

    print(f"epoch {epoch} | test RMSE: {rmse_on(test_loader):.4f}")


epoch 1 | test RMSE: 1.2155
epoch 2 | test RMSE: 1.1419
epoch 3 | test RMSE: 1.0907
epoch 4 | test RMSE: 1.0530
epoch 5 | test RMSE: 1.0240
epoch 6 | test RMSE: 1.0007
epoch 7 | test RMSE: 0.9819
epoch 8 | test RMSE: 0.9664
epoch 9 | test RMSE: 0.9536


In [25]:
import torch.onnx
import json

model.eval()

dummy_user = torch.zeros(1, dtype=torch.long).to(device)
dummy_movie = torch.zeros(1, dtype=torch.long).to(device)

torch.onnx.export(
    model,
    (dummy_user, dummy_movie),
    "recommender.onnx",
    input_names=["user_idx", "movie_idx"],
    output_names=["rating_pred"],
    dynamic_axes={
        "user_idx": {0: "batch"},
        "movie_idx": {0: "batch"},
        "rating_pred": {0: "batch"},
    },
    opset_version=17,
)

metadata = {
    "n_users": n_users,
    "n_movies": n_movies,
    "user2idx": user2idx,
    "movie2idx": movie2idx,
}

with open("metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

print("Saved recommender.onnx and metadata.json")


C:\Users\diego\AppData\Local\Temp\ipykernel_29428\937555015.py:9: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0214 10:14:54.692000 29428 .venv\Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0214 10:14:55.269000 29428 .venv\Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0214 10:14

[torch.onnx] Obtain model graph for `MF([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MF([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


C:\Python314\Lib\copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
C:\Users\diego\PycharmProjects\Notebooks\.venv\Lib\site-packages\torch\onnx\_internal\exporter\_onnx_program.py:460: UserWarning: # The axis name: batch will not be used, since it shares the same shape constraints with another axis: batch.
  rename_mapping = _dynamic_shapes.create_rename_mapping(
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Saved recommender.onnx and metadata.json
